In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import metricas_plots

In [ ]:
importlib.reload(metricas_plots)
from metricas_plots import PlotsMetricas, T, F
p = PlotsMetricas()

### Ler dados

In [ ]:
dados = pd.read_csv("dados/ariel_limpo_log10.csv.gz", compression="gzip")
larguras = p.targets[:4]
train, test = train_test_split(
    dados[p.features + larguras], test_size=0.25, random_state=4321
)
test.reset_index().tail()

### Fazendo amostra de um Latin Hypercube para redução de dimensionalidade

In [ ]:
train_lhc = p.lhs_subsample_with_stats(train, p.features, larguras, n=2000, k_neighbors=100)
test_lhc = p.lhs_subsample_with_stats(test, p.features, larguras, n=1000, k_neighbors=100)
test_lhc.tail()

In [ ]:
### Visualizar diagrama no espaço reduzido
# train_lhc[T.nii_ha.value] = train_lhc[T.nii.value+'_mean'].values - train_lhc[T.ha.value+'_mean'].values
# train_lhc[T.oiii_hb.value] = train_lhc[T.oiii.value+'_mean'].values - train_lhc[T.hb.value+'_mean'].values
# p.show_bpt(train_lhc, color=F.azmass.value, title="Latin Hypercube, N = 5000", densities=False)

## Treinar Operons para Média, Desvio Padrão e Covariâncias

In [ ]:
modelos = {}  # Dicionário para armazenar os modelos


def treinar_modelos_operon(hyper, X_train_bins, items, ignore_load=False):
    for model_key, file_name, col_name, label in items:
        if not ignore_load:
            modelo = p.load_operon(file_name)
        if ignore_load or modelo is None:
            print(f"Treinando {label}...")
            y = train_lhc[col_name].values.astype(np.float64)
            modelo = p.treinar_operon(hyper, X_train_bins, y)
            p.save_operon(modelo, file_name)
        else:
            print(f"Lido modelo salvo: {label}...")
        modelos[model_key] = modelo


def treinar_modelos_pysr(hyper, X_train_bins, items, ignore_load=False):
    for model_key, file_name, col_name, label in items:
        if not ignore_load:
            modelo = p.ler_pysr(hyper, file_name)
        if ignore_load or modelo is None:
            print(f"Treinando {label}...")
            y = train_lhc[col_name].values.astype(np.float64)
            modelo = p.treinar_pysr(hyper, X_train_bins, y)
        else:
            print(f"Lido modelo salvo: {label}...")
        modelos[model_key] = modelo

In [ ]:
# Selecionar feature para o treinamento
X_train_bins = train_lhc[p.features].values.astype(np.float64)

# Configuração do Operon
config_operon = {
    "random_state": 4321,
    "population_size": 2000,
    "generations": 2000,
    "allowed_symbols": "add,sub,mul,aq,constant,variable,pow,exp,tanh",
    "max_length": 25,
    "max_depth": 100,
    "optimizer_iterations": 1000,
    "model_selection_criterion": "bayesian_information_criterion",
    "objectives": ["r2", "length"],
    "n_threads": 12,
}

# Configuração do PySR
config_pysr = {
    "populations": 10,
    "population_size": 100,
    "binary_operators": ["+", "-", "*", "/"],
    "unary_operators": ["square", "exp", "tanh"],
    "maxdepth": 25,
    "model_selection": "accuracy",
    "parallelism": "multiprocessing",
    "procs": 12,
    "verbosity": 0,
}

# Modelos para os estimadores da Normal
items = (
    [
        (f"{l}_mean", f"{l}_all_mean", f"{l}_mean", f"MÉDIA para {l}") 
        for l in larguras
    ] + 
    [
        (f"{l}_std", f"{l}_all_std", f"{l}_std", f"DESVIO PADRÃO para {l}") 
        for l in larguras
    ] + 
    [
        (f"cov_{l1}_{l2}", f"{l1}_{l2}_all_cov", f"cov_{l1}_{l2}", f"COVARIÂNCIA para {l1} x {l2}")
        for l1, l2 in p.cov_pairs
    ]
)

modelo = "operon"
if modelo == "operon":
    treinar_modelos_operon(config_operon, X_train_bins, items)
    p.salva_equacoes_html(modelos, "n4d_all")
elif modelo == "pysr":
    treinar_modelos_pysr(config_pysr, X_train_bins, items)
else:
    raise ValueError("Modelo desconhecido. Use 'operon' ou 'pysr'!")

## Inspeção da qualidade dos modelos

In [ ]:
p.inspecao_modelos(modelos, train_lhc, p.features)
plt.savefig(f"results/regressoes/{modelo}_meanstds_n4d_all.png", bbox_inches="tight")
plt.close()

In [ ]:
p.inspecao_modelos_covs(modelos, train_lhc, p.features)
plt.savefig(f"results/regressoes/{modelo}_covs_n4d_all.png", bbox_inches="tight")
plt.close()

## Gerar Amostras do Conjunto de Teste com Normal Multivariada

In [ ]:
if (True):
    try:
        ler = pd.read_csv(f"results/amostras_all_{modelo}.csv")
        p_correcao = ler["p_correcao"].iloc[0]
        print(f"Amostras lidas do arquivo CSV para todas as features.")
    except:
        print("Gerando novas amostras!")
        p_correcao = p.gerar_amostras(modelos, modelo, test, test_lhc)
else:
    p_correcao = p.gerar_amostras(modelos, modelo, test, test_lhc)

In [ ]:
# Ler amostras geradas
df_amostras = pd.read_csv(f"results/amostras_all_{modelo}.csv")
df_amostras = df_amostras.dropna()
df_amostras = df_amostras.reset_index(drop=True)

# Calcular razões (já estão em log10, então é só subtrair)
test[T.nii_ha.value] = test[T.nii.value].values - test[T.ha.value].values
test[T.oiii_hb.value] = test[T.oiii.value].values - test[T.hb.value].values
df_amostras[T.nii_ha.value] = df_amostras[T.nii.value] - df_amostras[T.ha.value]
df_amostras[T.oiii_hb.value] = df_amostras[T.oiii.value] - df_amostras[T.hb.value]

df_amostras.tail()

### Histogramas marginais das amostras normais

In [ ]:
# Calcula divergência KL entre amostras e o conjunto observado
bins = np.linspace(-1, 2.5, 100)
targets = [T.nii, T.ha, T.oiii, T.hb]
kl_values = [p.calcula_kl(test, df_amostras, t.value, bins) for t in targets]

# Gráficos
fig, axs = plt.subplots(2, 2, figsize=(16, 12))
# plt.suptitle("Distribuição das amostras normais geradas em função das 6 features", fontsize="xx-large")

for i, (t, kl) in enumerate(zip(targets, kl_values)):
    ax = axs[i // 2][i % 2]
    p.histogram_v(
        test[t.value], "", ax, bins=bins, lim=(-1, 2.5), cor="darkblue", lbl="Test Set"
    )
    p.histogram_v(
        df_amostras[t.value],
        r"%s, $D_{KL}$ = %.4f" % (p.unidades[t.value], kl),
        ax,
        bins=bins,
        lim=(-1, 2.5),
        cor="lightgreen",
        lbl="Sample",
    )

plt.tight_layout()
plt.savefig(f"results/histogramas/marginais_amostras_n4d_all_{modelo}.png", bbox_inches="tight")

## Calcular razões do BPT

In [ ]:
# Matriz de correlação das amostras geradas
corr_gerada = spearmanr(df_amostras[larguras]).correlation
df_corrger = pd.DataFrame(corr_gerada, index=larguras, columns=larguras)

# Matriz de correlação dos dados reais (test set)
test_linhas = test[larguras].values
corr_real = spearmanr(test_linhas).correlation
df_corrtest = pd.DataFrame(corr_real, index=larguras, columns=larguras)

# Diferença total absoluta
diff_df = (df_corrger - df_corrtest).abs()
diff = diff_df.values[np.triu_indices(4, k=1)].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
p.plot_corr(
    corr_real,
    axes[0],
    larguras,
    title="Matriz de correlação dos dados de teste",
    type="inf",
)
p.plot_corr(
    corr_gerada,
    axes[1],
    larguras,
    title="Matriz de correlação das amostras normais,\n"
    + r"com $\Delta$=%.4f" % (diff),
    type="inf",
)
plt.savefig(
    f"results/correlacoes/corr_amostras_n4d_all_{modelo}.png", bbox_inches="tight"
)
plt.close()

### Diagramas de diagnóstico

In [ ]:
# Plotar com curvas de densidade
p.show_bpt(df_amostras, title="Estimadores Operon + Amostras Normal4D", densities=True, show=False)
plt.savefig(f"results/diagramas/bpt_densidades_amostras_n4d_all_{modelo}.png", bbox_inches="tight")

In [ ]:
# Plotar amostras geradas coloridas por alguma feature
p.show_bpt(df_amostras, F.azmass.value, title="Estimadores Operon + Amostras Normal4D", show=False)
plt.savefig(f"results/diagramas/bpt_cores_amostras_n4d_all_{modelo}.png", bbox_inches="tight")

In [ ]:
# Plotar com curvas de densidade
p.show_whan(df_amostras, title="Estimadores Operon + Amostras Normal4D", densities=True, show=False)
plt.savefig(f"results/diagramas/whan_densidades_amostras_n4d_all_{modelo}.png", bbox_inches="tight")

In [ ]:
p.show_whan(df_amostras, F.azmass.value, title="Estimadores Operon + Amostras Normal4D", show=False)
plt.savefig(f"results/diagramas/whan_cores_amostras_n4d_all_{modelo}.png", bbox_inches="tight")